# Multi-Layer Automotive Intrusion Detection System (IDS)
### Complete Implementation with Performance Visualizations

This notebook demonstrates the end-to-end pipeline for an automotive IDS, including results, training validation, and behavioral analysis plots as mandated by the project requirements.

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Add src to path
sys.path.append(os.getcwd())

from src.parser import CANParser
from src.features import CANFeatureEngineer
from src.validation import DataSplitter
from src.models import CANModelTrainer
from src.evaluation import IDSEvaluator
from src.telematics import TelematicsProcessor, BehavioralBaseline
from src.integration import AlertCorrelator
from src.plotting import IDSPlotter

print("Setup Complete. All Plotting Utilities Loaded.")

## 1. Data Exploration & Attack Analysis
We visualize the manifestations of a **DoS Attack** in the CAN timing domain.

In [ ]:
# Load and Preprocess sample
df = next(CANParser.load_csv('Data/DoS_dataset.csv', chunksize=20000))
df = CANParser.preprocess_df(df)
df = CANFeatureEngineer.extract_message_features(df)
df = CANFeatureEngineer.extract_window_features(df, window_size=50)

# Visualizing Injection Frequency
print("Analyzing DoS Attack Patterns...")
IDSPlotter.plot_attack_patterns(df.iloc[1000:5000], title="DoS Attack: CAN Message Delta-T Drops")

## 2. Model Training & Validation
Training an **XGBoost Classifier** and evaluating performance using Confusion Matrix and ROC Curves.

In [ ]:
X, y = CANFeatureEngineer.get_feature_matrix(df)
X_train, X_test, y_train, y_test = DataSplitter.temporal_split(X, y, test_size=0.2)

trainer = CANModelTrainer(model_type='xgboost')
trainer.train(X_train, y_train)

# Predictions for Evaluation
y_pred = trainer.predict(X_test)
y_probs = trainer.model.predict_proba(X_test)[:, 1]

# Performance Visuals
IDSPlotter.plot_confusion_matrix(y_test, y_pred, title="XGBoost IDS Confusion Matrix")
IDSPlotter.plot_roc_curve(y_test, y_probs, title="XGBoost IDS ROC Curve")

## 3. Telematics Behavioral Monitoring
Establishing a driving envelope and detecting deviations in vehicle speed/acceleration.

In [ ]:
norm_df = CANParser.parse_raw_text('Data/normal_run_data/normal_run_data.txt', max_lines=15000)
norm_df = CANParser.preprocess_df(norm_df)
tele_df = TelematicsProcessor.derive_features(norm_df)

baseline = BehavioralBaseline()
baseline.fit(tele_df)
anomalies = baseline.detect_anomalies(tele_df, threshold_sigma=3)

# Behavioral Visualization
IDSPlotter.plot_behavioral_envelope(tele_df, anomalies, col='Speed', title="Telematics Baseline: Normal Speed Envelope")
IDSPlotter.plot_behavioral_envelope(tele_df, anomalies, col='Acceleration', title="Telematics Baseline: Acceleration Profiling")

## 4. Integrated Multi-Layer Alert Analysis
Mapping correlated detections into vSOC alerts.

In [ ]:
can_alerts = pd.DataFrame({'Timestamp': [df['Timestamp'].iloc[-1]], 'Attack_Type': ['DoS']})
tele_anomalies = pd.DataFrame({'Timestamp': [df['Timestamp'].iloc[-1]], 'Anomaly_Score': [1]})

correlated = AlertCorrelator.correlate(can_alerts, tele_anomalies)
vsoc_alert = AlertCorrelator.generate_vsoc_alert(correlated.iloc[0])

print("Final Correlation Result:")
print(json.dumps(vsoc_alert, indent=2))